# Chapter 11 — The Smallest Useful Model Call

**Companion to *Applied AI*.**

This notebook accompanies Chapter 11. The chapter's fourth live OpenCode run is
preserved in this repository, and it is the best single artifact in the book:
the record's *observations* were accurate and its *interpretation* was wrong.

This notebook opens that artifact.

## Question

**Why must task, call and attempt identities stay separate — and what can you
recover from a preserved observation that a status field has already
mislabelled?**

## What this notebook establishes

- The three identities, read out of a real recorded call, shown to be distinct.
- A call recorded as `succeeded` whose preserved provider body says
  `finish_reason: "length"` — **the truncated answer the chapter found by
  reading bytes after the fact**.
- The usage the record kept (two numbers) beside the usage the provider
  actually reported (five fields, including 192 cached input tokens and a
  reasoning count of zero next to 1,128 characters of reasoning).
- A demonstration of one call with two attempts, and why unknown usage must
  propagate to an unknown total rather than to a number.

## What this notebook does **not** establish

- This is **one live call to one model on one day**. It shows what the runtime
  recorded, not how any model behaves.
- Reading `finish_reason` here does **not** re-run the call. The chapter is
  explicit that the request was paid for and cannot be repeated meaningfully.
- The chapter's other three live runs failed at the transport or gateway layer;
  their misclassifications are described in the chapter and only summarised
  here.

## Setup

In [1]:
import json
import os
from pathlib import Path

def find_evidence_dir(marker="ch11-live-opencode"):
    env = os.environ.get("APPLIED_AI_EVIDENCE")
    if env and Path(env).expanduser().is_dir():
        return Path(env).expanduser()
    here = Path.cwd().resolve()
    for base in (here, *here.parents):
        for cand in (base / "evidence",
                     base / "experiments" / "applied-ai" / "evidence"):
            if (cand / marker).is_dir():
                return cand
    raise FileNotFoundError(
        "Preserved evidence not found. Set APPLIED_AI_EVIDENCE to the "
        "directory holding the Applied AI evidence bundles."
    )

EVIDENCE_DIR = find_evidence_dir()
BUNDLE = EVIDENCE_DIR / "ch11-live-opencode"
events = json.loads((BUNDLE / "events.json").read_text(encoding="utf-8"))
artifact_paths = sorted((BUNDLE / "artifacts").glob("*.json"))

print("bundle :", BUNDLE.name)
print("events :", len(events))
print("artifacts:", [p.stem[:16] + "..." for p in artifact_paths])

bundle : ch11-live-opencode
events : 6
artifacts: ['1a459bea7ba6b472...']


## 1. What a recorded call actually wrote

Six events, in write order. Notice that **intent is written before the
request** — `call.manifest` precedes `attempt.started`.

In [2]:
for i, e in enumerate(events, 1):
    print(f"{i}. {e['kind']:<20} actor={e.get('actor_id','')}")

print()
print("The write order is the design: manifest (intent) before attempt")
print("(effect), observation after. Between attempt.started and the")
print("observation, the honest answer to 'did the provider serve it?' is")
print("'unknown'.")

1. task.created         actor=runtime
2. call.requested       actor=reviewer-live
3. call.manifest        actor=reviewer-live
4. attempt.started      actor=reviewer-live
5. attempt.completed    actor=reviewer-live
6. call.completed       actor=reviewer-live

The write order is the design: manifest (intent) before attempt
(effect), observation after. Between attempt.started and the
observation, the honest answer to 'did the provider serve it?' is
'unknown'.


## 2. Three identities that must never collapse

In [3]:
by_kind = {e["kind"]: e["payload"] for e in events}

task_id    = by_kind["task.created"]["task_id"]
call_id    = by_kind["call.manifest"]["call_id"]
attempt_id = by_kind["attempt.started"]["attempt_id"]

print(f"task_id    : {task_id}")
print(f"call_id    : {call_id}")
print(f"attempt_id : {attempt_id}")
print()
assert task_id != call_id != attempt_id
assert len({task_id, call_id, attempt_id}) == 3
print("assertion held: task_id != call_id != attempt_id")
print()
print("Each answers a different question:")
print("  task    - what are we trying to get done?")
print("  call    - what did we decide to ask, of which chamber?")
print("  attempt - what happened on the wire, this time?")

task_id    : task-live-1
call_id    : call-live-1
attempt_id : 0f3861fb-8b7b-475b-8f12-bbbc66e5da08

assertion held: task_id != call_id != attempt_id

Each answers a different question:
  task    - what are we trying to get done?
  call    - what did we decide to ask, of which chamber?
  attempt - what happened on the wire, this time?


## 3. The manifest: intent, frozen before any attempt

In [4]:
m = by_kind["call.manifest"]
for k in ("chamber", "requested_model", "resolved_model_id", "provider",
          "provider_revision", "pricing_version", "prompt_hash"):
    v = m.get(k)
    shown = "UNKNOWN (and recorded as unknown, not as a date)" if v is None else v
    if isinstance(shown, str) and len(shown) > 50:
        shown = shown[:46] + "..."
    print(f"  {k:<20} {shown}")

print()
print("requested parameters:", json.dumps(m.get("requested_parameters", {})))
print("effective parameters:", json.dumps(m.get("effective_parameters", {})))

  chamber              deep-review
  requested_model      review
  resolved_model_id    mimo-v2.5
  provider             opencode
  provider_revision    UNKNOWN (and recorded as unknown, not as a date)
  pricing_version      2026-09-01
  prompt_hash          3dfb44d00ca9ed65a29cc8dfc71b886a8fa63a79f19712...

requested parameters: {"max_tokens": 256}
effective parameters: {"endpoint": "/v1/chat/completions", "gateway": "opencode", "max_tokens": 256, "model": "mimo-v2.5", "protocol": "chat_completions"}


`provider_revision` is `None`. The chapter's rule:

```text
unknown  !=  absent  !=  zero  !=  inferred
```

A timestamp is not a model version, so nothing was invented.

## 4. What the record concluded

In [5]:
att = by_kind["attempt.completed"]
call = by_kind["call.completed"]

print(f"attempt status : {att.get('status')}")
print(f"error_kind     : {att.get('error_kind')}")
print(f"usage          : {json.dumps(att.get('usage'))}")
print(f"cost_usd       : {att.get('cost_usd')}   (cost_source: {att.get('cost_source')})")
print(f"call status    : {call.get('status')}")
print()
print("The recorded verdict on this call is: succeeded.")

attempt status : succeeded
error_kind     : None
usage          : {"input_tokens": 264, "output_tokens": 256, "source": "measured"}
cost_usd       : None   (cost_source: unknown)
call status    : succeeded

The recorded verdict on this call is: succeeded.


## 5. Now open the preserved observation

This is the move the whole chapter is built to make possible. The bytes were
stored **before anything parsed them**, so a later reader can check the
interpretation.

In [6]:
obs = json.loads(artifact_paths[0].read_text(encoding="utf-8"))
provider = obs["provider_response"]
choice = provider["choices"][0]

print("provider_call_id :", obs.get("provider_call_id"))
print("model            :", obs.get("model"))
print("protocol         :", obs.get("protocol"))
print()
print("finish_reason    :", repr(choice["finish_reason"]))

assert choice["finish_reason"] == "length"
print()
print("assertion held: the provider said the generation hit its token limit")

provider_call_id : gen-1789285773-Hs04dOHXGznryY8zu4BY
model            : mimo-v2.5
protocol         : chat_completions

finish_reason    : 'length'

assertion held: the provider said the generation hit its token limit


In [7]:
text = choice["message"]["content"]
print("the answer the record filed, last 90 characters:")
print()
print("   ..." + repr(text[-90:]))
print()
print(f"characters returned : {len(text)}")
print(f"output tokens       : {provider['usage']['completion_tokens']}")
print(f"max_tokens requested: {obs['effective_parameters'].get('max_tokens')}")
print()
print("The review stops mid-sentence. It was recorded as a success.")

the answer the record filed, last 90 characters:

   ...'ce, context, or qualifications.\n\n**Why it’s unsupported:**\n- **Absolute language** ("every'

characters returned : 246
output tokens       : 256
max_tokens requested: 256

The review stops mid-sentence. It was recorded as a success.


## Observation

Put the two side by side. Both are true statements about the same call.

In [8]:
print(f"{'source':<26}{'says'}")
print("-" * 70)
print(f"{'the runtime record':<26}call status: succeeded")
print(f"{'the preserved bytes':<26}finish_reason: length (truncated mid-sentence)")
print()
print("At this stage CodeAI read no finish or stop reason anywhere, so a")
print("truncated answer and a complete one were recorded identically.")
print()
print("Because the OBSERVATION was kept apart from the INTERPRETATION, the")
print("interpretation can be corrected without rewriting history. That is what")
print("the attempt's normalizer_version is for:")
print("   normalizer_version =", obs.get("normalizer_version"))

source                    says
----------------------------------------------------------------------
the runtime record        call status: succeeded
the preserved bytes       finish_reason: length (truncated mid-sentence)

At this stage CodeAI read no finish or stop reason anywhere, so a
truncated answer and a complete one were recorded identically.

Because the OBSERVATION was kept apart from the INTERPRETATION, the
interpretation can be corrected without rewriting history. That is what
the attempt's normalizer_version is for:
   normalizer_version = codeai-normalizer-v1


## 6. The usage the record did not keep

In [9]:
canonical = obs["usage"]                      # what the runtime kept
reported  = provider["usage"]                 # what the provider actually sent

print("canonical usage recorded by the runtime (v1, two numbers):")
print("  ", json.dumps(canonical))
print()
print("usage the provider actually reported:")
print(json.dumps(reported, indent=2))

cached = reported["prompt_tokens_details"]["cached_tokens"]
reasoning_tokens = reported["completion_tokens_details"]["reasoning_tokens"]
reasoning_text = choice["message"].get("reasoning") or ""

print()
print(f"cached input tokens        : {cached} of {reported['prompt_tokens']}")
print(f"=> fresh input             : {reported['prompt_tokens'] - cached}")
print(f"reported reasoning tokens  : {reasoning_tokens}")
print(f"reasoning characters present: {len(reasoning_text)}")
print(f"gateway 'cost' field        : {provider.get('cost')!r}  (a string, not a price)")

canonical usage recorded by the runtime (v1, two numbers):
   {"input_tokens": 264, "output_tokens": 256, "source": "measured"}

usage the provider actually reported:
{
  "completion_tokens": 256,
  "completion_tokens_details": {
    "audio_tokens": 0,
    "reasoning_tokens": 0
  },
  "prompt_tokens": 264,
  "prompt_tokens_details": {
    "audio_tokens": 0,
    "cache_write_tokens": 0,
    "cached_tokens": 192
  },
  "total_tokens": 520
}

cached input tokens        : 192 of 264
=> fresh input             : 72
reported reasoning tokens  : 0
reasoning characters present: 1128
gateway 'cost' field        : '0'  (a string, not a price)


Three facts in that block, each of which Chapter 13 turns into a rule:

- **192 of 264 input tokens were served from cache.** Cached and uncached
  tokens are priced an order of magnitude apart, so a "total tokens" figure was
  never a cost proxy.
- **`reasoning_tokens` is 0 beside 1,128 characters of reasoning.** A
  provider's own accounting breakdown is an observation to be interpreted, not
  a fact to be believed.
- **`"cost": "0"`** appears on subscription models and free models alike, so on
  its own it is not a price. The record keeps cost as `unknown` rather than
  importing it.

## 7. What the other three live runs recorded

The chapter's four runs were four separate recorded calls in four separate
workspaces — **experiments, not retries**. Calling them attempts would break
the definition this chapter's accounting depends on.

In [10]:
RUNS = [
    (1, "Responses",        "HTTP 403, Cloudflare 1010", "authentication_error",
     "an edge block; the credential was never evaluated"),
    (2, "Responses",        "HTTP 500",                  "provider_error",
     "wrong dialect: mimo-v2.5 is served on Chat Completions"),
    (3, "Chat Completions", "HTTP 400 MissingSessionID", "provider_error",
     "OUR request was incomplete - a 400 blames the client"),
    (4, "Chat Completions", "HTTP 200 with text",        "succeeded",
     "truncated: finish_reason=length (shown above)"),
]
print(f"{'run':<5}{'protocol':<19}{'what came back':<28}{'recorded as':<22}what it was")
print("-" * 118)
for n, proto, got, rec, actual in RUNS:
    print(f"{n:<5}{proto:<19}{got:<28}{rec:<22}{actual}")

print()
print("Observations accurate. Interpretations wrong on runs 1, 3 and 4.")
print("All three were caught in the preserved raw observations.")

run  protocol           what came back              recorded as           what it was
----------------------------------------------------------------------------------------------------------------------
1    Responses          HTTP 403, Cloudflare 1010   authentication_error  an edge block; the credential was never evaluated
2    Responses          HTTP 500                    provider_error        wrong dialect: mimo-v2.5 is served on Chat Completions
3    Chat Completions   HTTP 400 MissingSessionID   provider_error        OUR request was incomplete - a 400 blames the client
4    Chat Completions   HTTP 200 with text          succeeded             truncated: finish_reason=length (shown above)

Observations accurate. Interpretations wrong on runs 1, 3 and 4.
All three were caught in the preserved raw observations.


## 8. Demonstration: one call, two attempts

The chapter's defect: a client that retries an empty response, returns one
result, and reports one usage figure. The caller thinks it paid once.

In [11]:
from dataclasses import dataclass, field
from typing import Optional
import uuid

@dataclass
class Attempt:
    attempt_id: str
    call_id: str
    attempt_index: int
    status: str
    input_tokens: Optional[int]
    output_tokens: Optional[int]
    usage_source: str          # "measured" | "unavailable"

@dataclass
class Call:
    call_id: str
    task_id: str
    attempts: list = field(default_factory=list)

    def total_input(self):
        """UNKNOWN if any attempt's usage is unknown. Never zero-filled."""
        if any(a.usage_source == "unavailable" for a in self.attempts):
            known = sum(a.input_tokens or 0 for a in self.attempts)
            return None, f"unknown (at least {known} observed)"
        return sum(a.input_tokens for a in self.attempts), "measured"

task = "task-001"
call = Call(call_id="call-001", task_id=task)
call.attempts.append(Attempt(str(uuid.uuid4())[:8], "call-001", 1,
                             "transient_failure", None, None, "unavailable"))
call.attempts.append(Attempt(str(uuid.uuid4())[:8], "call-001", 2,
                             "succeeded", 264, 41, "measured"))

for a in call.attempts:
    print(f"  attempt {a.attempt_index}: {a.status:<20} "
          f"usage={a.usage_source:<13} in={a.input_tokens}")

total, note = call.total_input()
print()
print(f"provider requests actually made : {len(call.attempts)}")
print(f"results returned to the caller  : 1")
print(f"call total_input_tokens         : {total}  ->  {note}")
print()
assert total is None
print("assertion held: one unknown attempt makes the call total unknown")

  attempt 1: transient_failure    usage=unavailable   in=None
  attempt 2: succeeded            usage=measured      in=264

provider requests actually made : 2
results returned to the caller  : 1
call total_input_tokens         : None  ->  unknown (at least 264 observed)

assertion held: one unknown attempt makes the call total unknown


## Interpretation

The chapter's rule about missing data is the one that pays here. Usage goes
missing most often on **failed** attempts — timeouts, rejected requests,
dropped connections. Fill those gaps with zero and every cost-per-call figure
is biased downward, worst exactly where failures are most frequent.

In [12]:
naive = sum(a.input_tokens or 0 for a in call.attempts)       # zero-filled
print(f"zero-filled total   : {naive}  <- looks precise, understates the work")
print(f"honest total        : {total}  (lower bound {naive})")
print()
print("A retry is another ATTEMPT beneath the same CALL. Collapse the")
print("identities and 'how many times did we actually ask?' has no answer")
print("that does not require reading adapter source.")

zero-filled total   : 264  <- looks precise, understates the work
honest total        : None  (lower bound 264)

A retry is another ATTEMPT beneath the same CALL. Collapse the
identities and 'how many times did we actually ask?' has no answer
that does not require reading adapter source.


## Where this record is still weak

The chapter lists these, and the notebook can confirm two of them directly.

In [13]:
weaknesses = [
    ("Completion is not recorded",
     f"finish_reason present in bytes: {choice['finish_reason']!r}, "
     f"error_kind recorded: {att.get('error_kind')!r}"),
    ("Classification is unversioned interpretation",
     "two of four live runs were misclassified from the status code alone"),
    ("Identity is only unique within a ledger",
     f"call id is hand-assigned: {call_id!r} - merge two ledgers and it collides"),
    ("Cost cannot represent a subscription",
     f"cost_usd={att.get('cost_usd')}, cost_source={att.get('cost_source')!r}"),
]
for name, note in weaknesses:
    print(f"- {name}\n    {note}")

- Completion is not recorded
    finish_reason present in bytes: 'length', error_kind recorded: None
- Classification is unversioned interpretation
    two of four live runs were misclassified from the status code alone
- Identity is only unique within a ledger
    call id is hand-assigned: 'call-live-1' - merge two ledgers and it collides
- Cost cannot represent a subscription
    cost_usd=None, cost_source='unknown'


## Try it yourself

1. **Write the corrected interpreter.** Map `finish_reason` to
   `complete / truncated / filtered / unknown` and re-read the preserved bytes.
   You have just done Chapter 17's reinterpretation by hand — with no provider
   call and no change to history.
2. **Compute the cost two ways.** Price this call by total tokens, then price
   it with the 192 cached tokens charged at a tenth. How far apart are they?
   Chapter 13 is about why you cannot do this from a field name.
3. **Break the identity rule.** Merge two copies of this bundle and look for
   `call-live-1` in both. What breaks first?
4. **Find the orphan.** Delete `attempt.completed` from a copy of the events and
   re-read. The manifest and `attempt.started` remain: the effect is *unknown*,
   not absent. Chapter 16 is about what a process may safely do next.